# Model微调实例

## Step1 导包

In [1]:
import pandas as pd
from transformers import AutoTokenizer,AutoModelForSequenceClassification

## Step2 数据读取

In [2]:
data = pd.read_csv("./waimai_10k.csv")
data#查看数据的前n行，默认是前五行

,label,review
0,1,很快，好吃，味道足，量大
1,1,没有送水没有送水没有送水
2,1,非常快，态度好。
3,1,方便，快捷，味道可口，快递给力
4,1,菜味道很棒！送餐很及时！
...,...,...
11982,0,以前几乎天天吃，现在调料什么都不放，
11983,0,昨天订凉皮两份，什么调料都没有放，就放了点麻油，特别难吃，丢了一份，再也不想吃了
11984,0,"凉皮太辣,吃不下都"
11985,0,本来迟到了还自己点！！！


In [3]:
data.dropna()
data


,label,review
0,1,很快，好吃，味道足，量大
1,1,没有送水没有送水没有送水
2,1,非常快，态度好。
3,1,方便，快捷，味道可口，快递给力
4,1,菜味道很棒！送餐很及时！
...,...,...
11982,0,以前几乎天天吃，现在调料什么都不放，
11983,0,昨天订凉皮两份，什么调料都没有放，就放了点麻油，特别难吃，丢了一份，再也不想吃了
11984,0,"凉皮太辣,吃不下都"
11985,0,本来迟到了还自己点！！！


## Step3 创建dataset

In [4]:
from torch.utils.data import Dataset
#pytorch中提供的是一个抽象类，用户需要自己创建自己的数据集继承这个（或其他类型的数据集）
#一般来说需要用户自己实现的方法有以下几类
class Mydataset(Dataset):
    
    def __init__(self) -> None:
        super().__init__()
        #super()返回一个代理对象，在单继承中直接指向父类；在多继承中能正确地按顺序调用所有父类
        self.data = pd.read_csv("./waimai_10k.csv")
        self.data = self.data.dropna()

    #__gititem__()获取单数据
    def __getitem__(self, index):
        return self.data.iloc[index]["review"],self.data.iloc[index]["label"]

    #__len__()返回长度
    def __len__(self):
        return len(self.data)
#普通方法：需要你主动调用。比如你定义了 def train(self):，必须写 model.train() 才会执行。
# 魔法方法：不需要你主动调用，而是在特定场景下由 Python 解释器自动触发。
# 当你写 len(dataset) 时，Python 自动去调用 dataset.__len__()。
# 当你写 dataset[0] 时，Python 自动去调用 dataset.__getitem__(0)。
# 当你写 print(obj) 时，Python 自动去调用 obj.__str__()。


In [5]:
dataset = Mydataset()
for i in range(5):#不包含5
    print(dataset[i])


('很快，好吃，味道足，量大', np.int64(1))
('没有送水没有送水没有送水', np.int64(1))
('非常快，态度好。', np.int64(1))
('方便，快捷，味道可口，快递给力', np.int64(1))
('菜味道很棒！送餐很及时！', np.int64(1))


## Step4 拆分数据集

In [6]:
from torch.utils.data import random_split
#random_split本生是一个函数
trainset,validset = random_split(dataset,lengths=[0.9,0.1])
len(trainset),len(validset)

(10789, 1198)

## Step5 创建dataloader

In [7]:
#自己创建collate_fu函数
# collate_fu函数始终接收的是：dataset__getitem__的返回值根据据batch_size打包为一个batch列表
# 在python的底层会将return a,b打包为一个元组

import torch

tokenizer = AutoTokenizer.from_pretrained("rbt3")
def collate_fu(batch):
    texts,labels = [],[]
    for item in batch:
        texts.append(item[0])
        labels.append(item[1])
    inputs = tokenizer(texts,padding = "max_length",max_length = 128,truncation=True,return_tensors="pt")
    #   转换为张量
    inputs["labels"] = torch.tensor(labels)
    return inputs
    

In [8]:
from torch.utils.data import DataLoader
trainloader = DataLoader(trainset,batch_size=32,shuffle=True,collate_fn=collate_fu) #shuffle这个参数是是否打乱，batch批次是一次训练的数量
validloader = DataLoader(validset,batch_size=320,shuffle=False,collate_fn=collate_fu)
next(enumerate(trainloader))[1]
# next(iter(validloader))[0]#这里的[1]是在访问获得的元组中的第二个元素

#iter()函数是将一个可迭代对象打包为一个迭代器：直接获取下一个元素
#enumerate()函数是将打包为一个枚举对象：即自动对其中的元素进行编号始终产出一个（索引，元素）的长度为二的元组


# # DataLoader没有实现__getitem__()这个魔法方法，所以不是一个序列，无法直接使用trainloader[0]来访问（因为要实现shuffle这个方法）但是这是一个可迭代对象，所以可以通过迭代器来访问
# # enumerate(trainloader)
# # Python 内置的 enumerate() 将一个可迭代对象（这里是 trainloader）包装成一个枚举对象。
# # 每迭代一次，它会生成一个形如 (index, batch) 的元组，其中 index 是该批次的序号（从 0 开始），batch 就是该批次的数据。

# # next(...)
# # Python 内置函数，用于从迭代器中取出下一个元素。
# # 在这里，它从 enumerate 中取出第一个元素，即 (0, first_batch)。

# #由于lable本身就是一个数字，所以可以直接聚合为tensor类型的数据，但是由于review是一个句子，所以无法自动聚合为tensor形式



{'input_ids': tensor([[ 101,  679, 2582,  ...,    0,    0,    0],
        [ 101, 1962, 1391,  ...,    0,    0,    0],
        [ 101, 6206,  749,  ...,    0,    0,    0],
        ...,
        [ 101, 4696, 2552,  ...,    0,    0,    0],
        [ 101, 8108, 4157,  ...,    0,    0,    0],
        [ 101, 1922, 2714,  ...,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0,
        1, 0, 0, 0, 1, 0, 0, 0])}

## Step6 创建模型与优化器

In [9]:
from torch.optim import Adam
model = AutoModelForSequenceClassification.from_pretrained("rbt3")
# if torch.cuda.is_available():
#     model = model.cuda
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
# model.parameters()模型的参数
optimizer = Adam(model.parameters(),lr=2e-5)


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: rbt3
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training o

## Step7 训练模型

In [10]:
# 另一种写法
# def evaluate():
#     model.eval()
#     acc_num = 0
#     # 禁用梯度计算，让测试跑的更快
#     with torch.inference_mode():
#         for batch in validloader:
#             if torch.cuda.is_available():
#                 batch = {k: v.cuda() for k, v in batch.items()}
#             output = model(**batch)
#             # output.logits是原始预测分数，形状为 [batch_size, num_labels]
#             # dim=-1指在张量的最后（最里层）一个维度上进行操作
#             # torch.argmax()是获得最大值的索引
#             pred = torch.argmax(output.logits, dim=-1)
#             acc_num += (pred.long() == batch["labels"].long()).float().sum()
#         return acc_num / len(validset)

In [11]:
# 所有模型都有两种状态.eval()测试状态和.train()训练状态
# 当进行训练状态时，Dropouth会被激活，在前向传播中会随机关闭一些神经元，防止过拟合
# 测试函数
def evaluate():
    model.eval()
    correct = 0
    total = 0
    with torch.inference_mode():
        for batch in validloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}
            output = model(**batch)
            pred = torch.argmax(output.logits, dim=-1)  # [batch_size]
            labels = batch["labels"]                    # 注意键名是 "labels"
            correct += (pred == labels).sum().item()    # 累加正确预测的个数
            total += labels.size(0)                     # 累加总样本数
    return correct / total  # 返回准确率

# 写一下训练函数，其中的参数有epoch训练总轮次，log_step打印一次的步数
def train(epoch=3,log_step=100):
    # 总训练步数
    global_step = 0
    # 开始训练
    for ep in range(epoch):
        model.train()
        #batch是一个字典：其中含有input_ids,token_type_ids,attention_mask,lables
        for batch in trainloader:
            # 张量和模型都有to()方法可以转移设备
            batch = {k : v.to(device) for k , v in batch.items()}
            # 梯度清零
            optimizer.zero_grad()
            # 前向传播
            outputs = model(**batch)
            # 反向传播
            outputs.loss.backward()
            # 更新参数
            optimizer.step()
            if global_step % log_step == 0:
                    print(f"ep: {ep}, global_step: {global_step}, loss: {outputs.loss.item()}")
            global_step += 1
        acc = evaluate()
        print(f"ep:{ep},acc:{acc}")

In [12]:
# batch = next(enumerate(trainloader))[1]
# batch = {k : v.to(device) for k , v in batch.items()}
# output = model(**batch)
# output
# # 此时为预测模式，所以不会返回loss

In [13]:
train()

ep: 0, global_step: 0, loss: 0.729891300201416
ep: 0, global_step: 100, loss: 0.3257066309452057
ep: 0, global_step: 200, loss: 0.37365075945854187
ep: 0, global_step: 300, loss: 0.10179546475410461
ep:0,acc:0.8973288814691152
ep: 1, global_step: 400, loss: 0.13816174864768982
ep: 1, global_step: 500, loss: 0.07564166188240051
ep: 1, global_step: 600, loss: 0.18691276013851166
ep:1,acc:0.9098497495826378
ep: 2, global_step: 700, loss: 0.29737067222595215
ep: 2, global_step: 800, loss: 0.15124158561229706
ep: 2, global_step: 900, loss: 0.06493973731994629
ep: 2, global_step: 1000, loss: 0.11622282862663269
ep:2,acc:0.9073455759599333


## Step8 测试模型

In [14]:
sen = "这家饭店的鱼很好吃，价格也很公道，以后常来"
id2_label = {0:"差评！",1:"好评！"}
with torch.inference_mode():
    input = tokenizer(sen,return_tensors="pt")
    input = {k:v.cuda() for k , v in input.items()}
    output = model(**input)
    logits = output.logits
    pred = torch.argmax(logits,dim = -1)
    print(f"输入：{sen}\n模型预测结果：{id2_label.get(pred.item())}")

输入：这家饭店的鱼很好吃，价格也很公道，以后常来
模型预测结果：好评！


## Step9 使用pipe组件直接测试

In [15]:
from transformers import pipeline 

model.config.id2label = id2_label
pipe = pipeline(task = "text-classification",model = model,tokenizer=tokenizer,device=0)
pipe(sen)

[{'label': '好评！', 'score': 0.9210253953933716}]